# Density-driven demand — proof of concept

Makes `density` drive the demand **level** (population), not only the daily shape.

`volume = anchor x income_factor(income) x density_factor(density)`

`density_factor` is *derived* from `unit_share_by_density` + `units_per_building` (no new free
parameter), and dialled by `gamma`: `0.0` = today's behaviour, `1.0` = full slot-conservation.

Nothing in `src/` is modified — only `build_portfolios` and `evolve_assignments` are overridden here.

## 0. Setup

In [1]:
%matplotlib inline
import copy, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parents[1]          # notebooks/prospection -> repo root

from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
bootstrap_project(ROOT)
with KedroSession.create(project_path=ROOT) as session:
    ctx = session.load_context()
    CATALOG, PARAMS = ctx.catalog, ctx.params

WN_RAW    = CATALOG.load("graeme_network")
DISTRICTS = CATALOG.load("districts")
NODE2DIST = {n: d for d, ns in DISTRICTS["districts"].items() for n in ns}
len(NODE2DIST), list(DISTRICTS["districts"])

[08/28/26 18:37:46] INFO     Using                                                                  ]8;id=416903;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=139405;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\framework\project\__init__.py#302\302]8;;\
                             'c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\framewo                
                             rk\project\rich_logging.yml' as logging configuration.                                

[08/28/26 18:37:55] INFO     No typed parameter requirements found, returning original   ]8;id=881513;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\validation\parameter_validator.py\parameter_validator.py]8;;\:]8;id=282659;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\validation\parameter_validator.py#124\124]8;;\
                             parameters                                                                            

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=635629;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=828853;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro_telemetry\plugin.py#273\273]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[08/28/26 18:37:56] INFO     Loading data from graeme_network (WntrNetworkDataset)...          ]8;id=370168;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=288050;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\io\data_catalog.py#1050\1050]8;;\

                    INFO     Loading data from districts (YAMLDataset)...                      ]8;id=520765;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=777864;file://c:\Users\arthu\anaconda3\envs\TsLeaks\lib\site-packages\kedro\io\data_catalog.py#1050\1050]8;;\

(113, ['District_A', 'District_B', 'District_C', 'District_D', 'District_E'])

In [2]:
from fedwater.pipelines.network_prep.nodes import configure_network, apply_coupling
from fedwater.pipelines.urban_scenario.nodes import (
    build_income_factors, build_drift_schedule, SECONDS_PER_DAY, ANCHOR_DAYS_PER_MONTH)
from fedwater.pipelines.demand_synthesis.nodes import synthesize_demands, apply_drift_ramp
from fedwater.pipelines.hydraulics.nodes import run_hydraulics
from fedwater.experiments.spec import decode_map

## 1. Density factors, derived

`units_per_slot(d) = 1 / sum_t ( unit_share[t] / mean_units_per_building[t] )` — the unit-share
weighted harmonic mean of units per building, i.e. expected occupancy of one building slot.
Assuming a block has a fixed number of slots, the level factor is the ratio of occupancies.

In [3]:
def units_per_slot(share: dict, upb: dict) -> float:
    return 1.0 / sum(s / np.mean(upb[t]) for t, s in share.items())


def build_density_factors(buildings: dict, gamma: float = 1.0,
                          reference: str = "low") -> pd.DataFrame:
    ups = {d: units_per_slot(sh, buildings["units_per_building"])
           for d, sh in buildings["unit_share_by_density"].items()}
    ref = ups[reference]
    return pd.DataFrame([{"density": d, "units_per_slot": u,
                          "density_factor": (u / ref) ** gamma} for d, u in ups.items()])


INC = build_income_factors(PARAMS["buildings"])
display(INC)
pd.concat([build_density_factors(PARAMS["buildings"], g).assign(gamma=g)
           for g in (0.0, 0.5, 1.0)], ignore_index=True)

,income,mean_unit_m3_month,income_factor
0,low,1.45,0.527273
1,medium,2.75,1.000000
2,high,7.05,2.563636


,density,units_per_slot,density_factor,gamma
0,low,3.360000,1.000000,0.0
1,medium,4.801444,1.000000,0.0
2,high,9.013894,1.000000,0.0
3,low,3.360000,1.000000,0.5
4,medium,4.801444,1.195408,0.5
5,high,9.013894,1.637897,0.5
6,low,3.360000,1.000000,1.0
7,medium,4.801444,1.429001,1.0
8,high,9.013894,2.682706,1.0


## 2. POC overrides

In [4]:
def _cohorts(node, district, income, density, anchor, volume, mean_unit, buildings, rng):
    total_units, out = volume / mean_unit, []
    for tpl, share in buildings["unit_share_by_density"][density].items():
        units = float(total_units * share)
        if units < 0.5:
            continue
        lo, hi = buildings["units_per_building"][tpl]
        out.append({"node": node, "district": district, "income": income,
                    "density": density, "template": tpl, "units": units,
                    "n_buildings": max(1, int(round(units / rng.uniform(lo, hi)))),
                    "anchor_m3_month": anchor, "volume_m3_month": volume})
    return out


def poc_build_portfolios(wn, districts, income_factors, density_factors, scenario,
                         buildings, hydraulics, seed):
    rng = np.random.default_rng(seed)
    mapping = dict(zip(districts["districts"].keys(), scenario["income_density_mapping"]))
    f_inc  = dict(zip(income_factors["income"], income_factors["income_factor"]))
    mean_u = dict(zip(income_factors["income"], income_factors["mean_unit_m3_month"]))
    f_den  = dict(zip(density_factors["density"], density_factors["density_factor"]))

    rows = []
    for district, nodes in districts["districts"].items():
        income, density = mapping[district]
        for node in nodes:
            base_si = wn.get_node(node).demand_timeseries_list[0].base_value
            anchor  = base_si * hydraulics["anchor_scale"] * SECONDS_PER_DAY * ANCHOR_DAYS_PER_MONTH
            volume  = anchor * f_inc[income] * f_den[density]      # <-- the only change
            rows += _cohorts(node, district, income, density, anchor, volume,
                             mean_u[income], buildings, rng)

    df = pd.DataFrame(rows)
    per_node = df.drop_duplicates("node")
    calib = per_node["anchor_m3_month"].sum() / per_node["volume_m3_month"].sum()
    df["calibration"] = calib
    df["volume_m3_month"] *= calib
    df["units"] *= calib
    return df

In [5]:
def poc_evolve_assignments(portfolios_t0, drift_schedule, income_factors, density_factors,
                           buildings, scenario, seed):
    """Per-row to_income/to_density, so several districts can drift differently."""
    rng = np.random.default_rng(seed)
    f_inc  = dict(zip(income_factors["income"], income_factors["income_factor"]))
    mean_u = dict(zip(income_factors["income"], income_factors["mean_unit_m3_month"]))
    f_den  = dict(zip(density_factors["density"], density_factors["density_factor"]))
    ds = drift_schedule.set_index("node")
    calib = portfolios_t0["calibration"].iloc[0]

    frames = []
    for month in range(scenario["n_months"]):
        snap = portfolios_t0.copy()
        switch = ds.index[ds["drift_month"] <= month].tolist()
        if switch:
            base, new = snap[snap["node"].isin(switch)].drop_duplicates("node"), []
            for _, r in base.iterrows():
                to_i, to_d = ds.loc[r["node"], "to_income"], ds.loc[r["node"], "to_density"]
                vol = r["anchor_m3_month"] * calib * f_inc[to_i] * f_den[to_d]
                new += _cohorts(r["node"], r["district"], to_i, to_d,
                                r["anchor_m3_month"], vol, mean_u[to_i], buildings, rng)
            new = pd.DataFrame(new).assign(calibration=calib)
            snap = pd.concat([snap[~snap["node"].isin(switch)], new], ignore_index=True)
        frames.append(snap.assign(month=month))
    return pd.concat(frames, ignore_index=True)

## 3. World runner

`drifts` is a list — one entry per drifting district, so single- and multi-client drift use the
same call. `seed_node=None` asks the engine to auto-pick the district's largest-demand junction.

In [ ]:
CFG = dict(n_months=10, days_per_month=15, resolution_h=1,
           anchor_scale=0.05, gamma=1.0, drift_ramp_days=60,
           warmup_months=2, sim_seed=42)


def run_world(map_code: str, drifts: list[dict], **over):
    cfg = {**CFG, **over}
    p    = copy.deepcopy(PARAMS)
    time = {k: cfg[k] for k in ("n_months", "days_per_month", "resolution_h")}
    hyd  = {**p["hydraulics"], "anchor_scale": cfg["anchor_scale"]}
    pat  = {**p["patterns"],   "drift_ramp_days": cfg["drift_ramp_days"]}
    scen = {**p["scenario"], "n_months": cfg["n_months"],
            "income_density_mapping": decode_map(map_code)}

    wn = configure_network(copy.deepcopy(WN_RAW), hyd, time)
    wn, _ = apply_coupling(wn, DISTRICTS, p["coupling"], seed=cfg["sim_seed"])

    den = build_density_factors(p["buildings"], cfg["gamma"])
    pf  = poc_build_portfolios(wn, DISTRICTS, INC, den, scen,
                               p["buildings"], hyd, cfg["sim_seed"])

    sched = pd.concat([
        build_drift_schedule(
            wn, DISTRICTS,
            {**scen, "drift": {**scen["drift"], **d, "seed_node": None,
                               "warmup_months": cfg["warmup_months"]}},
            seed=cfg["sim_seed"] + i)
        for i, d in enumerate(drifts)], ignore_index=True)

    tl  = poc_evolve_assignments(pf, sched, INC, den, p["buildings"], scen, cfg["sim_seed"])
    dem = apply_drift_ramp(synthesize_demands(tl, pat, time, cfg["sim_seed"]), sched, pat, time)
    pres, flows, _ = run_hydraulics(wn, dem)

    return dict(map=map_code, drifts=drifts, gamma=cfg["gamma"], cfg=cfg, time=time,
                calibration=float(pf["calibration"].iloc[0]), schedule=sched,
                timeline=tl, demand=dem, pressures=pres, flows=flows)

In [10]:
BASE_DRIFT = [{"tgt_district": "District_D", "to_income": "low", "to_density": "high"}]

res_on  = run_world("LL_LM_LH_LL_LL", BASE_DRIFT, seed_node = 2, gamma=1.0)   # density moves level
res_off = run_world("LL_LM_LH_LL_LL", BASE_DRIFT, seed_node = 2, gamma=0.0)   # today's behaviour (control)

print(f"calibration  gamma=1: {res_on['calibration']:.3f}   gamma=0: {res_off['calibration']:.3f}")
res_on["schedule"].head()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 BASE_DRIFT = [{"tgt_district": "District_D", "to_income": "low", "to_density": "high"}]      │
│   2                                                                                              │
│ ❱ 3 res_on  = run_world("LL_LM_LH_LL_LL", BASE_DRIFT, seed_node = 2, gamma=1.0)   # density      │
│   4 res_off = run_world("LL_LM_LH_LL_LL", BASE_DRIFT, seed_node = 2, gamma=0.0)   # today's      │
│   5                                                                                              │
│   6 print(f"calibration  gamma=1: {res_on['calibration']:.3f}   gamma=0: {res_off['calibrati     │
│                                                                                                  │
│ in run_world:22                                                                                  │
│                                                                                                  │
│   19 │   pf  = poc_build_portfolios(wn, DISTRICTS, INC, den, scen,                               │
│   20 │   │   │   │   │   │   │      p["buildings"], hyd, cfg["sim_seed"])                        │
│   21 │                                                                                           │
│ ❱ 22 │   sched = pd.concat([                                                                     │
│   23 │   │   build_drift_schedule(                                                               │
│   24 │   │   │   wn, DISTRICTS,                                                                  │
│   25 │   │   │   {**scen, "drift": {**scen["drift"], **d, "seed_node": None,                     │
│                                                                                                  │
│ in <listcomp>:23                                                                                 │
│                                                                                                  │
│   20 │   │   │   │   │   │   │      p["buildings"], hyd, cfg["sim_seed"])                        │
│   21 │                                                                                           │
│   22 │   sched = pd.concat([                                                                     │
│ ❱ 23 │   │   build_drift_schedule(                                                               │
│   24 │   │   │   wn, DISTRICTS,                                                                  │
│   25 │   │   │   {**scen, "drift": {**scen["drift"], **d, "seed_node": None,                     │
│   26 │   │   │   │   │   │   │      "warmup_months": cfg["warmup_months"]}},                     │
│                                                                                                  │
│ C:\Users\arthu\USPy\10_Mestrado\experiments\fedWater\src\fedwater\pipelines\urban_scenario\nodes │
│ .py:124 in build_drift_schedule                                                                  │
│                                                                                                  │
│   121 │   G = wn.to_graph().to_undirected().subgraph(nodes)                                      │
│   122 │   seed_node = str(drift["seed_node"])                                                    │
│   123 │   if seed_node not in nodes:                                                             │
│ ❱ 124 │   │   raise ValueError(f"seed_node {seed_node} not in {district}")                       │
│   125 │                                                                                          │
│   126 │   drifted = {seed_node: drift["warmup_months"]}                                          │
│   127 │   frontier = {seed_node}                           

## 4. Before / during / after

In [ ]:
def _cols(res):     return [c for c in res["demand"].columns if c != "month"]
def _steps(res):    return int(round(24 / res["time"]["resolution_h"]))
def _tgt(res):      return res["drifts"][0]["tgt_district"]
def _onset(res):    return int(res["schedule"]["drift_month"].min())
def _end(res):      return int(res["schedule"]["drift_month"].max())


def phases(res):
    o, e, n = _onset(res), _end(res), res["time"]["n_months"]
    return {"before": max(o - 1, 0), "during": min((o + e) // 2 + 1, n - 1), "after": n - 1}


def mean_day(res, nodes, month):
    d = res["demand"]
    s = d.loc[d["month"] == month, nodes].sum(axis=1).to_numpy()
    return s.reshape(-1, _steps(res)).mean(0)

In [ ]:
def plot_levels(res, ax=None):
    d, ax = res["demand"], ax or plt.subplots(figsize=(9, 3))[1]
    m = d.groupby("month")[_cols(res)].mean()
    for dist, nodes in DISTRICTS["districts"].items():
        ns = [n for n in nodes if n in m.columns]
        ax.plot(m.index, m[ns].sum(axis=1), marker="o", ms=3, label=dist,
                lw=2.2 if dist == _tgt(res) else 1.0,
                alpha=1.0 if dist == _tgt(res) else 0.45)
    ax.axvspan(_onset(res), _end(res), color="grey", alpha=0.15, zorder=0)
    ax.set(xlabel="month", ylabel="district demand (L/s)",
           title=f"{res['map']}  ->  {_tgt(res)} density-high   (gamma={res['gamma']})")
    ax.legend(fontsize=7, ncol=5)


fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
plot_levels(res_on, axes[0]); plot_levels(res_off, axes[1]); plt.tight_layout()

In [ ]:
def plot_diurnal(res, ax=None):
    ax = ax or plt.subplots(figsize=(5, 3.2))[1]
    nodes = [n for n in DISTRICTS["districts"][_tgt(res)] if n in res["demand"].columns]
    t = (np.arange(_steps(res)) + .5) * res["time"]["resolution_h"]
    for name, month in phases(res).items():
        ax.plot(t, mean_day(res, nodes, month), label=f"{name} (m{month})")
    ax.set(xlabel="hour", ylabel="L/s", title=f"{_tgt(res)} mean day — gamma={res['gamma']}")
    ax.legend(fontsize=8)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
plot_diurnal(res_on, axes[0]); plot_diurnal(res_off, axes[1]); plt.tight_layout()

In [ ]:
def plot_pressure(res, ax=None):
    ax = ax or plt.subplots(figsize=(9, 3))[1]
    p = res["pressures"]
    junc = [c for c in p.columns if c in NODE2DIST]
    p = p.assign(month=res["demand"]["month"].to_numpy())
    for dist, nodes in DISTRICTS["districts"].items():
        ns = [n for n in nodes if n in junc]
        ax.plot(p.groupby("month")[ns].min().min(axis=1), marker="o", ms=3, label=dist,
                lw=2.2 if dist == _tgt(res) else 1.0,
                alpha=1.0 if dist == _tgt(res) else 0.45)
    ax.axvspan(_onset(res), _end(res), color="grey", alpha=0.15, zorder=0)
    ax.axhline(5.0, color="crimson", ls="--", lw=1)      # V3 hard floor
    ax.axhline(10.0, color="orange", ls=":", lw=1)       # NBR band floor
    ax.set(xlabel="month", ylabel="min pressure (mca)", title=f"gamma={res['gamma']}")
    ax.legend(fontsize=7, ncol=5)


fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, sharey=True)
plot_pressure(res_on, axes[0]); plot_pressure(res_off, axes[1]); plt.tight_layout()

## 5. Sweep — initial maps x drift specs x gamma

In [ ]:
def metrics(res):
    ph, out = phases(res), {"map": res["map"], "gamma": res["gamma"],
                            "drift": "+".join(d["tgt_district"][-1] +
                                              d["to_income"][0].upper() +
                                              d["to_density"][0].upper()
                                              for d in res["drifts"]),
                            "calibration": res["calibration"]}
    d, p = res["demand"], res["pressures"]
    tgt_nodes = sorted({n for x in res["drifts"]
                        for n in DISTRICTS["districts"][x["tgt_district"]]
                        if n in d.columns})
    junc = [c for c in p.columns if c in NODE2DIST]
    p = p.assign(month=d["month"].to_numpy())

    def blk(month, nodes):
        s = d.loc[d["month"] == month, nodes].sum(axis=1).to_numpy()
        return s.mean(), s.reshape(-1, _steps(res)).max(1).mean()

    (mb, pb), (ma, pa) = blk(ph["before"], tgt_nodes), blk(ph["after"], tgt_nodes)
    out |= {"d_mean_%": 100 * (ma / mb - 1), "d_peak_%": 100 * (pa / pb - 1),
            "pmin_before": p.loc[p["month"] == ph["before"], junc].to_numpy().min(),
            "pmin_after":  p.loc[p["month"] == ph["after"],  junc].to_numpy().min(),
            "pmin_all":    p[junc].to_numpy().min()}
    out["V3_pass"] = out["pmin_all"] >= 5.0
    return out


MAPS = ["LL_LL_LL_LL_LL", "LL_LM_LH_LL_LL", "LM_LM_LM_LM_LM"]
DRIFTS = {
    "D: LL->LH": [{"tgt_district": "District_D", "to_income": "low",  "to_density": "high"}],
    "D: LL->HH": [{"tgt_district": "District_D", "to_income": "high", "to_density": "high"}],
    "A+D: LL->LH": [{"tgt_district": "District_A", "to_income": "low", "to_density": "high"},
                    {"tgt_district": "District_D", "to_income": "low", "to_density": "high"}],
}
GAMMAS = [0.0, 1.0]

In [ ]:
rows, worlds = [], {}
for mp, (dname, dr), g in itertools.product(MAPS, DRIFTS.items(), GAMMAS):
    key = (mp, dname, g)
    try:
        r = run_world(mp, dr, gamma=g)
        worlds[key] = r
        rows.append({**metrics(r), "drift": dname})
    except Exception as e:
        rows.append({"map": mp, "drift": dname, "gamma": g, "error": type(e).__name__})
        print("FAIL", key, e)

sweep = pd.DataFrame(rows)
sweep.round(2)

In [ ]:
piv = sweep.pivot_table(index=["map", "drift"], columns="gamma",
                        values=["d_mean_%", "d_peak_%", "pmin_after"])
display(piv.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, col, lbl in zip(axes, ["d_mean_%", "pmin_after"],
                        ["target-district mean demand change (%)", "min pressure after drift (mca)"]):
    w = sweep.pivot_table(index=["map", "drift"], columns="gamma", values=col)
    w.plot.bar(ax=ax, width=.8, legend=(col == "d_mean_%"))
    ax.set(title=lbl, xlabel="")
    ax.tick_params(axis="x", labelsize=7, rotation=60)
axes[1].axhline(5.0, color="crimson", ls="--", lw=1)
axes[1].axhline(10.0, color="orange", ls=":", lw=1)
plt.tight_layout()

## 6. Read-out

- `gamma=0` reproduces today's simulator: a pure `LL -> LH` drift is volume-neutral, so
  `d_mean_% ~ 0` and `pmin_after` is flat or slightly *higher* (flatter profile, lower peak).
- `gamma=1` makes the same drift a population change: mean and peak demand rise, pressure falls
  in the drifting district.
- Watch `pmin_all` / `V3_pass`: if `LL -> HH` at `anchor_scale=0.05` breaks the 5 mca floor,
  re-run the sweep with a lower `anchor_scale`, or an intermediate `gamma`, before touching `src/`.
- `calibration` stays near 1 when the initial map is density-mixed; a uniform-`LL` map with
  `gamma=1` shifts it, which is expected (calibration scales population, not per-unit thirst).